## ⚙️ Environment Setup (run once)

A maioria dos serviços AgentCore (Gateway, Memory, Runtime, Registry) requer
versões recentes de `boto3`. Esta cislula instala/atualiza tudo o que o
workshop precisa.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 09.1 — Launch Portal and Run Journeys

## Overview

🎉 **Lab final.** Subir o portal (servidor HTTP stdlib) + Server-Sent Events e ver tudo funcionando
ponta a ponta:

- Login Ana e Carlos via Cognito
- SmartAgent → Specialists → Gateway → Cedar → Lambda
- Memory STM/LTM
- Guardrails
- Observability

## Prerequisites

- ✅ Labs 01-05 (mínimo) — todo o stack precisa estar deployado
- ✅ Recomendado: Lab 06 (Guardrails), 07 (Registry), 08 (Observability)

## Step 1: Verificar que stack is completo

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config

cfg = load_config()

required = [
    "COGNITO_USER_POOL_ID",
    "COGNITO_CLIENT_ID",
    "GATEWAY_ID",
    "MEMORY_ID",
    "RUNTIME_SMART_AGENT_ARN",
    "RUNTIME_GRID_AGENT_ARN",
]
missing = [k for k in required if not cfg.get(k)]
if missing:
    print(f"⚠️  Variáveis ausentes em config.env: {missing}")
    print("   Volte aos labs anteriores antes de continuar.")
else:
    print("✓ Stack completo, prosseguir.")

## Step 2: Subir o portal

O portal já vem em `ui/server.py` (servidor HTTP da biblioteca padrão do
Python — sem dependências extras). Ele lê o `config.env` e expõe:

- `/` → página inicial; `/assistente` → chat conversacional (SSE)
- `/governanca` → decisões Cedar e mistricas; `/sishasa` → health check
- APIs: `POST /api/auth/login`, `POST /api/agent/invoke`,
  `GET /api/syshas/status`, `GET /api/governance/cedar-decisions`

In [ ]:
print("Para subir o portal (em um terminal):")
print("  cd 09-End-to-End-with-UI")
print("  python3 ui/server.py            # porta 8080 (use --port para mudar)")
print("\nPortal disponível em: http://localhost:8080")
print("\nLogin Ana:    ana.operadora@workshop.local / Workshop@2025!")
print("Login Carlos: carlos.gestor@workshop.local / Workshop@2025!")

## Step 3: Rodar as jornadas pris-definidas

### Jornada A — Ana (operadora) consulta rede
- Login com Ana
- "Como is o east sector?" → SmartAgent → GridMonitorAgent → Gateway → Cedar permit P1 → grid_api
- Resposta com dados reais

### Jornada B — Ana tenta aprovar (DENY P3)
- "Aprove a work order WO-2024-0041"
- SmartAgent → MaintenanceAgent → Gateway → Cedar **DENY P3** (forbid operators)
- Specialist devolve mensagem amigável

### Jornada C — Carlos aprova (PERMIT P2)
- Logout, login com Carlos
- "Aprove a work order WO-2024-0041"
- Cedar **PERMIT P2** → Lambda invocada → status=approved

### Jornada D — Cedar P8 context-based
- Como Ana: "Crie work order priority high para SE-LESTE-03"
- Cedar P8 dispara DENY com base em `context.input.priority`
- Como Carlos a mesma chamada → PERMIT

## ✅ Validation

Após cada jornada, abra o **CloudWatch Logs** em `/aws/spans` e veja:
- AuthorizeAction com decision PERMIT/DENY
- Tool name, principal, action, resource
- Determining policies (qual P0-P8 decidiu)

## 🎓 O que você construiu (no workshop completo)

- ✅ **Identity** — Cognito User Pool with 2 groups + 2 users
- ✅ **Gateway** — MCP-fy de 5 Lambdas com JWT auth
- ✅ **Policy** — 9 Cedar policies com P0-P8 cobrindo identity-based + context-based
- ✅ **Memory** — STM + LTM + semantic search com isolamento multi-tenant
- ✅ **Runtime** — 6 agentes Strands (1 router + 5 specialists)
- ✅ **Guardrails** — PII + prompt injection + topics
- ✅ **Registry** — catálogo com workflow de aprovação DRAFT/PENDING_APPROVAL/APPROVED
- ✅ **Observability** — CloudTrail + spans + 8 alarmes

## 🧹 Cleanup completo

```bash
python -m shared.utils.cleanup --all
```

## 📚 Nexts passos

- **Adapte para outro setor:** create `shared/<asset>/<setor>/` para healthcare ou financial
- **Customize policies:** edite os `.cedar` e re-run o Lab 03
- **Mude o prompt do SmartAgent:** edite `shared/prompts/<setor>/smart_agent.md`

🎉 **Workshop concluído!**